# Persian SEA — single-video pilot in VS Code


In [1]:
import importlib.util, os, platform, sys
from pathlib import Path

IS_COLAB = importlib.util.find_spec("google.colab") is not None
VIDEO_ID = "news001"  # Change if file names differ
print("Runtime:", "COLAB (REMOTE GPU)" if IS_COLAB else "LOCAL (MAC)")
print("Python:", sys.version.split()[0], "System:", platform.system(), "cwd:", Path.cwd())
if IS_COLAB:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

Runtime: COLAB (REMOTE GPU)
Python: 3.13.15 System: Linux cwd: /content
CUDA available: True
GPU: Tesla T4


## 1 — MAC ONLY: light checks (no model downloads, no DINOv2)


In [2]:
if IS_COLAB:
    print("SKIPPED: local check only. Switch to Mac .venv kernel.")
else:
    import subprocess
    loc = Path.cwd().resolve()
    candidates = [loc, *loc.parents, loc / "Lab_project/sea_starter",
                  loc.parent / "Lab_project/sea_starter"]
    ROOT = next((c for c in candidates if (c / "pyproject.toml").is_file()
                 and (c / "src/persian_sea").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Open the project in VS Code and set cwd to sea_starter / project root")
    print("Project root:", ROOT)
    subprocess.run([sys.executable, str(ROOT / "scripts/pilot_workflow.py"),
                    "check", "--id", VIDEO_ID, "--config", "configs/pilot_mac.yaml"],
                   cwd=ROOT, check=True)

SKIPPED: local check only. Switch to Mac .venv kernel.


## 2 — MAC ONLY: SEA temporal alignment (no GPU)



In [3]:
if IS_COLAB:
    print("SKIPPED: SEA local only")
else:
    import subprocess
    assert "ROOT" in globals(), "Run local check first"
    subprocess.run([sys.executable, str(ROOT / "scripts/pilot_workflow.py"),
                    "sea", "--id", VIDEO_ID, "--config", "configs/pilot_mac.yaml"],
                   cwd=ROOT, check=True)

SKIPPED: SEA local only


## 3 — MAC ONLY: create one small transport ZIP



In [4]:
if IS_COLAB:
    print("SKIPPED: transport bundle must be created locally")
else:
    import subprocess
    assert "ROOT" in globals(), "Run local check first"
    DRIVE_SYNC_FOLDER = ""  
    
    output_zip = (Path(DRIVE_SYNC_FOLDER).expanduser() / "news001_inputs.zip"
                  if DRIVE_SYNC_FOLDER.strip() else ROOT / "news001_inputs.zip")
    subprocess.run([sys.executable, str(ROOT / "scripts/pilot_workflow.py"),
                    "bundle", "--id", VIDEO_ID, "--config", "configs/pilot_mac.yaml",
                    "--output", str(output_zip)], cwd=ROOT, check=True)
    print("ZIP:", output_zip)
    print("For permanent access: sync this ZIP to MyDrive/sign_pilot/news001_inputs.zip")

SKIPPED: transport bundle must be created locally


---

## 4 — COLAB GPU ONLY: remote machine setup

Now select **Colab → Auto Connect** in the kernel selector (official Google Colab VS Code extension). Rerun cell **0**. **Never run local stages 1–3 on Colab.**

`ZIP_SOURCE = "drive"` is the persistent option. First sync `news001_inputs.zip` to `MyDrive/sign_pilot` on your Mac. Authorization for Drive mounting is still required. If you instead uploaded the ZIP into the live Colab server via VS Code's Explorer, set `ZIP_SOURCE = "uploaded"` and adjust `COLAB_UPLOADED_ZIP` if needed.

In [5]:
if not IS_COLAB:
    print("SKIPPED: select a Colab GPU kernel, then rerun cell 0")
else:
    import shutil, subprocess, sys
    from pathlib import Path
    import torch
    assert torch.cuda.is_available(), "Please enable a GPU in Colab runtime"
    ZIP_SOURCE = "drive"  # 'drive' (persistent sync) or 'uploaded' (VS Code Upload to Colab)
    COLAB_UPLOADED_ZIP = Path("/content/news001_inputs.zip")
    if ZIP_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        INPUT_ZIP = Path("/content/drive/MyDrive/sign_pilot/news001_inputs.zip")
        OUTPUT_DRIVE = Path("/content/drive/MyDrive/sign_pilot/output_features")
    else:
        INPUT_ZIP = COLAB_UPLOADED_ZIP
        OUTPUT_DRIVE = None
    assert INPUT_ZIP.is_file(), f"Missing remote ZIP: {INPUT_ZIP}"
    PROJECT_PARENT = Path("/content")
    REMOTE_ROOT = PROJECT_PARENT / "research_sign/Lab_project/sea_starter"
    if not REMOTE_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        "https://github.com/Aihan-2005/research_sign.git",
                        str(PROJECT_PARENT / "research_sign")], check=True)
    subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                   cwd=REMOTE_ROOT, check=True)
    print("Remote root:", REMOTE_ROOT)

MessageError: [dfs_ephemeral] Credentials propagation unsuccessful: the server still reports the credentials as unauthorized

## 5 — COLAB GPU ONLY: install dependencies and extract the ZIP


In [ ]:
if not IS_COLAB:
    print("SKIPPED: Colab-only cell")
else:
    import hashlib, json, subprocess, sys, zipfile
    from pathlib import Path
    assert "REMOTE_ROOT" in globals(), "Run remote setup first"
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
                   cwd=REMOTE_ROOT, check=True)
    allowed_prefixes = ("data/videos/", "data/subtitles_raw/", "data/poses/",
                        "outputs/segmentation/", "outputs/text_features/")
    with zipfile.ZipFile(INPUT_ZIP) as zf:
        names = zf.namelist()
        for name in names:
            pp = Path(name)
            if pp.is_absolute() or ".." in pp.parts:
                raise ValueError(f"Unsafe path in ZIP: {name}")
            if name not in ("pilot_manifest.json", "data/pilot_ids.txt") and not name.startswith(allowed_prefixes):
                raise ValueError(f"Unexpected path in ZIP: {name}")
        manifest = json.loads(zf.read("pilot_manifest.json"))
        assert manifest["video_id"] == VIDEO_ID, "Wrong video in bundle"
        for f in manifest["files"]:
            data = zf.read(f["path"])
            assert len(data) == f["bytes"] and hashlib.sha256(data).hexdigest() == f["sha256"], f["path"]
        zf.extractall(REMOTE_ROOT)
    print("Verified and extracted:", len(manifest["files"]), "files")

## 6 — COLAB GPU ONLY: generate config and run only DINOv2 visual extraction


In [ ]:
if not IS_COLAB:
    print("SKIPPED: intentionally never run full DINOv2 on Mac")
else:
    import yaml, subprocess
    assert "REMOTE_ROOT" in globals(), "Run remote setup first"
    cfg_path = REMOTE_ROOT / "configs/pilot_remote_gpu.yaml"
    cfg = yaml.safe_load((REMOTE_ROOT / "configs/baseline.yaml").read_text(encoding="utf-8"))
    cfg["paths"]["video_ids"] = "data/pilot_ids.txt"
    cfg["pose"]["model_complexity"] = 1
    cfg["scanner"]["device"] = "cuda"
    cfg["scanner"]["rgb_batch_size"] = 8
    cfg["scanner"]["overwrite"] = False
    cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    subprocess.run(["persian-sea-preprocess", "--config", "configs/pilot_remote_gpu.yaml",
                    "--video-ids", "data/pilot_ids.txt", "--steps", "visual"],
                   cwd=REMOTE_ROOT, check=True)

## 7 — COLAB GPU ONLY: verify and save persistent outputs


In [ ]:
if not IS_COLAB:
    print("SKIPPED: Colab-only cell")
else:
    import shutil, subprocess
    report = REMOTE_ROOT / "outputs/reports/news001_final_gpu.json"
    subprocess.run(["persian-sea-inspect-cache", f"outputs/features/{VIDEO_ID}.npz"],
                   cwd=REMOTE_ROOT, check=True)
    subprocess.run(["persian-sea-preflight", "--config", "configs/pilot_remote_gpu.yaml",
                    "--video-ids", "data/pilot_ids.txt", "--check-pose", "--check-caches",
                    "--report", str(report)], cwd=REMOTE_ROOT, check=True)
    if OUTPUT_DRIVE is not None:
        OUTPUT_DRIVE.mkdir(parents=True, exist_ok=True)
        for source in (REMOTE_ROOT / f"outputs/features/{VIDEO_ID}.npz",
                       REMOTE_ROOT / f"outputs/features/{VIDEO_ID}.json", report):
            destination = OUTPUT_DRIVE / source.name
            shutil.copy2(source, destination)
            print("Saved to Drive:", destination)
    else:
        print("Copy/download files from:", REMOTE_ROOT / "outputs/features", "and", report)

## 8 — MAC ONLY, after the GPU output is synced back


In [ ]:
if IS_COLAB:
    print("SKIPPED: final check is on Mac. Switch back to local .venv kernel.")
else:
    import subprocess
    from pathlib import Path
    if "ROOT" not in globals():
        loc = Path.cwd().resolve()
        candidates = [loc, *loc.parents, loc / "Lab_project/sea_starter"]
        ROOT = next((c for c in candidates if (c / "pyproject.toml").is_file()
                     and (c / "src/persian_sea").is_dir()), None)
    assert ROOT is not None, "Open workspace at sea_starter and rerun local check"
    subprocess.run([sys.executable, str(ROOT / "scripts/pilot_workflow.py"),
                    "check", "--id", VIDEO_ID, "--require-visual"], cwd=ROOT, check=True)
    subprocess.run(["persian-sea-preflight", "--config", "configs/pilot_mac.yaml",
                    "--video-ids", "data/pilot_ids.txt", "--check-pose", "--check-caches",
                    "--report", "outputs/reports/news001_final_mac.json"], cwd=ROOT, check=True)

In [6]:
from pathlib import Path

p = Path("/content/news001_inputs.zip")

print("ZIP found:", p.is_file())
if p.is_file():
    print("Size MB:", round(p.stat().st_size / 1024**2, 2))


ZIP found: False
